In [4]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. KHAI BÁO ĐỊA CHỈ & ĐỌC DỮ LIỆU
# ==========================================
print("⏳ Đang tải dữ liệu từ ổ cứng...")
base_path = r"D:\Data set"

# Đọc 3 tập dữ liệu đã chia từ trước
train_df = pd.read_csv(f"{base_path}\\train.csv")
valid_df = pd.read_csv(f"{base_path}\\valid.csv")
test_df = pd.read_csv(f"{base_path}\\test.csv")

# Với mô hình học máy truyền thống, ta gộp Train và Valid lại thành một tập lớn 
# để mô hình có lượng dữ liệu từ vựng dồi dào nhất có thể
full_train_df = pd.concat([train_df, valid_df], ignore_index=True)

# ==========================================
# 2. TIỀN XỬ LÝ NHÃN (LABELS)
# ==========================================
print("⏳ Đang xử lý nhãn (Labels)...")

# Hàm tách chuỗi (VD: "PITCH, REJECTION") thành danh sách (['PITCH', 'REJECTION'])
def process_labels(text):
    if pd.isna(text): return []
    return [label.strip() for label in text.split(',')]

# Áp dụng hàm tách nhãn
full_train_df['call_code_list'] = full_train_df['call_code'].apply(process_labels)
test_df['call_code_list'] = test_df['call_code'].apply(process_labels)

# Chuyển đổi danh sách nhãn thành ma trận nhị phân [0, 1] cho máy tính hiểu
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(full_train_df['call_code_list'])
y_test = mlb.transform(test_df['call_code_list'])

print(f"✅ Đã nhận diện được {len(mlb.classes_)} nhãn gốc: {list(mlb.classes_)}")

# ==========================================
# 3. TRÍCH XUẤT ĐẶC TRƯNG VĂN BẢN (BAG OF WORDS)
# ==========================================
print("\n⏳ Đang đếm từ vựng trên tập Train...")
# Khởi tạo công cụ đếm. 
# ngram_range=(1, 2) nghĩa là đếm cả từ đơn ("fees") và cụm 2 từ ("hidden fees")
counter = CountVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')

# Tìm các từ khóa từ tập Train và đếm số lần xuất hiện
X_train_count = counter.fit_transform(full_train_df['call_transcript'])

# Dùng chính bộ từ vựng đã học được ở Train để đếm trên tập Test
X_test_count = counter.transform(test_df['call_transcript'])

print(f"👉 Mô hình đã lập được một từ điển gồm {X_train_count.shape[1]} từ khóa.")

# ==========================================
# 4. HUẤN LUYỆN MÔ HÌNH (TRAINING)
# ==========================================
print("\n🚀 Đang huấn luyện mô hình phân loại (Logistic Regression)...")
# Dùng class_weight='balanced' để mô hình tự động tăng trọng số cho các nhãn hiếm gặp
base_model = LogisticRegression(solver='liblinear', class_weight='balanced', random_state=42)
counting_model = OneVsRestClassifier(base_model)

# Tiến hành học (khớp ma trận từ vựng với ma trận nhãn)
counting_model.fit(X_train_count, y_train)

# ==========================================
# 5. ĐÁNH GIÁ TRÊN TẬP TEST
# ==========================================
print("\n📊 Đang chấm điểm mô hình trên tập Test...")
y_pred_count = counting_model.predict(X_test_count)

# In kết quả tổng quan
print("\n" + "="*50)
print("🏆 KẾT QUẢ MÔ HÌNH 'TẬP ĐẾM' (CountVectorizer + LR)")
print("="*50)
print(f"F1 MICRO    : {f1_score(y_test, y_pred_count, average='micro')*100:.2f}%")
print(f"F1 MACRO    : {f1_score(y_test, y_pred_count, average='macro')*100:.2f}%")
print(f"PRECISION   : {precision_score(y_test, y_pred_count, average='micro')*100:.2f}%")
print(f"RECALL      : {recall_score(y_test, y_pred_count, average='micro')*100:.2f}%")
print(f"EXACT MATCH : {accuracy_score(y_test, y_pred_count)*100:.2f}%")
print("="*50)

# In báo cáo phân tích sâu cho từng nhãn
print("\n📋 CHI TIẾT TỪNG NHÃN (CLASSIFICATION REPORT):")
print(classification_report(y_test, y_pred_count, target_names=mlb.classes_, zero_division=0))

⏳ Đang tải dữ liệu từ ổ cứng...
⏳ Đang xử lý nhãn (Labels)...
✅ Đã nhận diện được 32 nhãn gốc: ['ACTIVE_LISTENING', 'ANGRY_OUTBURST', 'ANNOYED_SIGHING', 'APATHETIC_RESPONSE', 'CLOSING_NEGOTIATION', 'COMPETITOR_COMPARISON', 'CURIOUS_EXPLORATION', 'DEFENSIVE_POSTURE', 'DEMANDING_MANAGER', 'DO_NOT_CALL_REQUEST', 'ENTHUSIASTIC_AGREEMENT', 'FEE_DISCUSSION', 'FOLLOW_UP_EMAIL_REQUESTED', 'FREQUENT_INTERRUPTIONS', 'HARD_REJECTION', 'INDECISIVE_FLIPPING', 'MISUNDERSTANDING', 'NEEDS_ANALYSIS', 'OBJECTION_HANDLING', 'OPENING', 'OVERWHELMED_CONFUSION', 'PASSIVE_AGREEMENT', 'PRODUCT_PITCH', 'RUSHING_THE_CALL', 'SARCASTIC_MOCKERY', 'SOFT_REJECTION', 'STALLING_FOR_TIME', 'SUCCESSFUL_SALE', 'SUDDEN_HANG_UP', 'SUSPICIOUS_PROBING', 'THREATENING_COMPLAINT', 'WARM_LEAD']

⏳ Đang đếm từ vựng trên tập Train...
👉 Mô hình đã lập được một từ điển gồm 5000 từ khóa.

🚀 Đang huấn luyện mô hình phân loại (Logistic Regression)...

📊 Đang chấm điểm mô hình trên tập Test...

🏆 KẾT QUẢ MÔ HÌNH 'TẬP ĐẾM' (CountVectoriz

In [11]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from tqdm import tqdm

# ==========================================
# 1. LOAD DATA
# ==========================================
base_path = r"D:\Data set"

train_df = pd.read_csv(f"{base_path}\\train.csv")
valid_df = pd.read_csv(f"{base_path}\\valid.csv")
test_df  = pd.read_csv(f"{base_path}\\test.csv")

def process_labels(text):
    if pd.isna(text): return []
    return [l.strip() for l in text.split(',')]

train_df['labels'] = train_df['call_code'].apply(process_labels)
valid_df['labels'] = valid_df['call_code'].apply(process_labels)
test_df['labels']  = test_df['call_code'].apply(process_labels)

# ==========================================
# 2. ENCODE LABELS (FIX FLOAT)
# ==========================================
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df['labels']).astype(np.float32)
y_valid = mlb.transform(valid_df['labels']).astype(np.float32)
y_test  = mlb.transform(test_df['labels']).astype(np.float32)

num_labels = len(mlb.classes_)
print(f"✅ Num labels: {num_labels}")

train_df['labels'] = list(y_train)
valid_df['labels'] = list(y_valid)
test_df['labels']  = list(y_test)

# ==========================================
# 3. DATASET
# ==========================================
train_dataset = Dataset.from_pandas(train_df[['call_transcript','labels']])
valid_dataset = Dataset.from_pandas(valid_df[['call_transcript','labels']])
test_dataset  = Dataset.from_pandas(test_df[['call_transcript','labels']])

# ==========================================
# 4. TOKENIZER
# ==========================================
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["call_transcript"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

# 👉 QUAN TRỌNG: không set columns để tránh lỗi dtype
train_dataset.set_format(type="torch")
valid_dataset.set_format(type="torch")
test_dataset.set_format(type="torch")

# ==========================================
# 5. MODEL
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ==========================================
# 6. METRICS
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()

    threshold = 0.5
    preds = (probs >= threshold).astype(int)

    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="micro"),
        "recall": recall_score(labels, preds, average="micro"),
        "accuracy": accuracy_score(labels, preds)
    }

# ==========================================
# 7. TRAINING ARGS (TRANSFORMERS 5.x)
# ==========================================
training_args = TrainingArguments(
    output_dir=f"{base_path}\\roberta_model",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    eval_strategy="epoch",       # ✅ đúng cho v5
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True
)

# ==========================================
# 8. CUSTOM TRAINER (FIX FLOAT)
# ==========================================
class MyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels").float()   # ép float
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ==========================================
# 9. TRAINER
# ==========================================
trainer = MyTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,   # ✅ transformers 5.x
    compute_metrics=compute_metrics
)

# ==========================================
# 10. TRAIN
# ==========================================
trainer.train()

# ==========================================
# 11. PREDICT (có tqdm)
# ==========================================
print("\n🚀 Predicting test set...")

preds = []
model.eval()

for batch in tqdm(test_dataset, desc="Predicting"):
    input_ids = batch['input_ids'].unsqueeze(0).to(device)
    attention_mask = batch['attention_mask'].unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    prob = torch.sigmoid(outputs.logits).cpu().numpy()[0]
    preds.append(prob)

preds = np.array(preds)

# ==========================================
# 12. EVALUATE
# ==========================================
threshold = 0.5
y_pred = (preds >= threshold).astype(int)

print("\n📊 RESULT")
print(f"F1 MICRO : {f1_score(y_test, y_pred, average='micro'):.4f}")
print(f"F1 MACRO : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"PRECISION: {precision_score(y_test, y_pred, average='micro'):.4f}")
print(f"RECALL   : {recall_score(y_test, y_pred, average='micro'):.4f}")
print(f"ACCURACY : {accuracy_score(y_test, y_pred):.4f}")

✅ Num labels: 32


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11950.79it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Precision,Recall,Accuracy
1,0.186428,0.175774,0.675763,0.429376,0.790941,0.589866,0.098152
2,0.157538,0.154339,0.723082,0.543323,0.807877,0.654396,0.150693
3,0.137988,0.148399,0.736653,0.572858,0.803273,0.680237,0.168591


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la


🚀 Predicting test set...


Predicting: 100%|██████████| 1733/1733 [01:09<00:00, 25.06it/s]


📊 RESULT
F1 MICRO : 0.7364
F1 MACRO : 0.5650
PRECISION: 0.8078
RECALL   : 0.6767
ACCURACY : 0.1766


In [7]:
import transformers
print("Transformers version:", transformers.__version__)
print("Transformers path:", transformers.__file__)

Transformers version: 5.3.0
Transformers path: c:\Users\quoct\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\__init__.py
